# Project 2 Milestone 3: UMAP Nonlinear Embedding and Candidate Structure Exploration

This notebook extends the PCA baseline from Project 2 Milestone 2 by applying UMAP to the combined Gaia–LAMOST chemo-kinematic feature space.

The goal is to examine whether Project 1 candidate stars occupy coherent or peripheral regions in a nonlinear low-dimensional embedding.

UMAP is used here as an exploratory visualization method. The resulting structures should not be interpreted as confirmed astrophysical substructures without further clustering, validation, and comparison with known Galactic components.

## Motivation

The PCA baseline showed that many Project 1 chemo-kinematic candidates lie outside the dense main stellar locus. This motivates a nonlinear embedding step to test whether the same candidates form clearer local structures in a UMAP representation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
import umap

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

In [ ]:
DATA_PATH = Path('../data/processed/gaia_lamost_larger_velocity_features.csv')
FIGURE_DIR = Path('../figures')
OUTPUT_DIR = Path('../data/processed')

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.shape

## Combined Chemo-Kinematic Feature Space

In [ ]:
combined_features = [
    'feh',
    'bp_rp',
    'absolute_g_mag',
    'tangential_velocity_kms',
    'rv',
    'galcen_vx_kms',
    'galcen_vy_kms',
    'galcen_vz_kms',
    'galcen_vtot_kms',
]

combined = df.dropna(subset=combined_features).copy()
combined.shape

## Standardization and UMAP Embedding

The features are standardized before UMAP so that quantities with large numerical ranges do not dominate the embedding.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(combined[combined_features])

reducer = umap.UMAP(
    n_neighbors=30,
    min_dist=0.10,
    n_components=2,
    metric='euclidean',
    random_state=42,
)

embedding = reducer.fit_transform(X_scaled)

combined['umap_1'] = embedding[:, 0]
combined['umap_2'] = embedding[:, 1]

combined[['umap_1', 'umap_2']].head()

## UMAP Colored by Metallicity

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    combined['umap_1'],
    combined['umap_2'],
    c=combined['feh'],
    s=18,
    alpha=0.8,
)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('Project 2 UMAP: combined chemo-kinematic space colored by [Fe/H]')
cbar = plt.colorbar(scatter)
cbar.set_label('[Fe/H]')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'project2_umap_combined_chemo_kinematic_by_feh.png', dpi=200)
plt.show()

## UMAP Colored by Galactocentric Total Velocity

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    combined['umap_1'],
    combined['umap_2'],
    c=combined['galcen_vtot_kms'],
    s=18,
    alpha=0.8,
)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('Project 2 UMAP: combined chemo-kinematic space colored by Galactocentric velocity')
cbar = plt.colorbar(scatter)
cbar.set_label('Galactocentric total velocity [km/s]')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'project2_umap_combined_chemo_kinematic_by_vtot.png', dpi=200)
plt.show()

## UMAP with Project 1 Candidate Markers

Candidate flags are used only for post-hoc interpretation.

In [ ]:
candidate_mask = combined['chemo_kinematic_candidate'].astype(bool)

plt.figure(figsize=(8, 6))
plt.scatter(
    combined.loc[~candidate_mask, 'umap_1'],
    combined.loc[~candidate_mask, 'umap_2'],
    s=16,
    alpha=0.5,
    label='Non-candidate',
)
plt.scatter(
    combined.loc[candidate_mask, 'umap_1'],
    combined.loc[candidate_mask, 'umap_2'],
    s=45,
    alpha=0.9,
    marker='x',
    label='Project 1 chemo-kinematic candidate',
)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('Project 2 UMAP with Project 1 candidate markers')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'project2_umap_combined_chemo_kinematic_candidates.png', dpi=200)
plt.show()

## Save UMAP Embedding

In [ ]:
umap_output_cols = [
    'source_id',
    'umap_1',
    'umap_2',
    'feh',
    'bp_rp',
    'absolute_g_mag',
    'tangential_velocity_kms',
    'rv',
    'galcen_vtot_kms',
    'metallicity_group',
    'high_vtan_candidate',
    'metal_poor_candidate',
    'chemo_kinematic_candidate',
]

UMAP_OUTPUT_PATH = OUTPUT_DIR / 'project2_combined_chemo_kinematic_umap_embedding.csv'
combined[umap_output_cols].to_csv(UMAP_OUTPUT_PATH, index=False)
UMAP_OUTPUT_PATH

## Milestone 3 Notes

- UMAP is used as an exploratory nonlinear embedding method.
- Project 1 candidate flags are used only for interpretation after embedding.
- Visual grouping in UMAP space does not by itself prove the existence of distinct astrophysical populations.
- Later milestones should apply density-based clustering and compare cluster membership with physical diagnostics.